# Assignment 4: Evidence-Based Policy from a Document Database
**Name(s):** Anupat Suttilert; Nongnapat Sunontanum  
**Student ID(s):** Anupat Suttilert - 6624650526; Nongnapat Sunontanum - 6624650302  
**Problem title:** ผู้สูงอายุที่ไม่พบบันทึกเบี้ยยังชีพ: ตรวจสอบสิทธิและเชื่อมต่อการช่วยเหลือเชิงรุก

DSI463 | Tasks 1–5 + Bonus A + Bonus B | Prepared 28 October 2026

**ข้อมูล MSO เป็นข้อมูลสังเคราะห์ทั้งหมด ตัวเลขจาก MSO ไม่ใช่สถิติจริงของประเทศไทย**
คำว่า “ช่องว่าง/ไม่พบบันทึก” ในงานนี้หมายถึงไม่พบ WFEL9 ไม่ใช่การยืนยันว่ามีสิทธิตามกฎหมายหรือไม่ได้รับเงินจริง

In [1]:
from datetime import datetime
from pathlib import Path
from pprint import pprint
import os, json, copy
import pandas as pd
from pymongo import MongoClient
from bson import json_util
from IPython.display import display, Markdown

# Default matches the supplied Docker compose. MONGO_URI can select another local instance or Atlas.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/mso_member.json').exists())
URI = os.environ.get('MONGO_URI', 'mongodb://127.0.0.1:27017')
db = MongoClient(URI, serverSelectionTimeoutMS=10000)['mso']
db.command('ping')
SNAPSHOT = datetime(2024, 6, 30)
AGE = {'$dateDiff': {'startDate': '$birthdate', 'endDate': SNAPSHOT, 'unit': 'year'}}
ACTIVE = {'status': {'$ne': 'removed'}, 'vital_status': 1}
pd.set_option('display.max_columns', 25)
pd.set_option('display.max_rows', 80)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
RESULTS = {}
def agg(coll, pipeline):
    return pd.DataFrame(list(db[coll].aggregate(pipeline)))
def show(key, pipeline, coll='mso_member'):
    frame = agg(coll, pipeline)
    RESULTS[key] = frame
    view = pd.json_normalize(frame.to_dict('records'))
    if len(view.columns) > 10:
        display(view.set_index('_id').T if '_id' in view.columns else view.T)
    else:
        display(view)
    return frame
def arr(path): return {'$ifNull': ['$' + path, []]}
def has(path, value): return {'$in': [value, arr(path)]}
def pct(num, den):
    return {'$cond': [{'$gt': [den, 0]}, {'$multiply': [100, {'$divide': [num, den]}]}, None]}
FLAGS = {'age': AGE, 'missing_allowance': {'$not': [has('welf_eldery.code', 'WFEL9')]}}
ELDERS = [{'$match': ACTIVE}, {'$set': FLAGS}, {'$match': {'age': {'$gte': 60}}}]
TARGET = ELDERS + [{'$match': {'missing_allowance': True}}]
HH_JOIN = [
    {'$lookup': {'from': 'mso_household', 'localField': 'hh_id', 'foreignField': '_id', 'as': 'hh'}},
    {'$unwind': {'path': '$hh', 'preserveNullAndEmptyArrays': True}},
    {'$match': {'hh.status': {'$ne': 'removed'}}}
]
def rate_group(key):
    return [
        {'$group': {'_id': key, 'elderly': {'$sum': 1}, 'gap': {'$sum': {'$cond': ['$missing_allowance', 1, 0]}}}},
        {'$set': {'gap_pct': pct('$gap', '$elderly')}}, {'$sort': {'gap_pct': -1, '_id': 1}}
    ]
counts = {c: db[c].count_documents({}) for c in ['mso_household', 'mso_family', 'mso_member']}
display(pd.DataFrame(list(counts.items()), columns=['collection', 'documents']))
assert list(counts.values()) == [2000, 2217, 7518], 'Run load_data.py first.'
runtime_info = {'mongodb_version':db.client.server_info()['version'], 'endpoint':str(db.client.address), 'platform':db.command('buildInfo').get('buildEnvironment',{}).get('target_os','unknown')}
print('MongoDB runtime:', runtime_info, '| Snapshot:', SNAPSHOT.date())
pass

,collection,documents
0,mso_household,2000
1,mso_family,2217
2,mso_member,7518


MongoDB runtime: {'mongodb_version': '7.0.45', 'endpoint': "('127.0.0.1', 27017)", 'platform': 'linux'} | Snapshot: 2024-06-30


## วิธีรันและกติกาการนับ
ติดตั้ง `pymongo pandas nbformat nbclient ipykernel` เปิด MongoDB และรัน `load_data.py` ก่อนเปิด notebook นี้
ทุก pipeline ที่นับคนเริ่มด้วย ACTIVE; ฝั่งครัวเรือนที่ join กรอง removed แยก โดยไม่ใส่ vital_status ซึ่งไม่มีใน household
ใช้ `$ifNull` ก่อนอ่าน array และแยก missing/unknown ออกจาก “ไม่มี” เมื่อเปรียบเทียบที่อยู่อาศัย
การไม่พบ WFEL9 อาจเกิดจากไม่ได้รับหรือไม่ได้บันทึก จึงเป็นรายชื่อเพื่อยืนยันสิทธิ ไม่ใช่รายชื่ออนุมัติจ่ายเงิน

**อายุ:** ใช้ `$dateDiff(..., unit='year')` ตาม setup ของโจทย์อย่างสม่ำเสมอ ซึ่งนับขอบเขตปีปฏิทิน
จึงไม่ใช่อายุครบวันเกิดอย่างละเอียด; ตรวจ sensitivity ด้วย birthdate cutoff วันที่ 30 มิ.ย. 1964 เพิ่มด้านล่าง
การเทียบอัตราแสดงความสัมพันธ์ ไม่พิสูจน์สาเหตุ ไม่ขยายผลจากตัวอย่างสังเคราะห์เป็นประชากรจริง

## Task 1 — Explore the database
### 1a. Structure

In [2]:
for collection in ['mso_household', 'mso_family', 'mso_member']:
    doc = db[collection].find_one()
    print('\n' + collection, '| top-level fields:', len(doc))
    # find_one() retrieves the document; show only linked and analytical paths to keep output compact.
    keep = ['_id', 'hh_id', 'fam_id', 'hh_type', 'fam_type', 'birthdate', 'province_code',
            'welf_eldery', 'prob_accessibility', 'prob_summary']
    pprint({k: doc[k] for k in keep if k in doc}, width=105)
pass


mso_household | top-level fields: 31
{'_id': ObjectId('647e3af1183724b80f000001'),
 'hh_type': {'code': 'HHT3', 'desc': 'ครัวเรือนขยาย (พ่อ แม่ ลูก พี่ น้อง ปู่ ย่า ตา ยาย หลาน)'},
 'province_code': '10'}

mso_family | top-level fields: 26
{'_id': ObjectId('647e3af2ce02a3d3e2000002'),
 'fam_type': {'code': 'FT9', 'desc': 'ครอบครัวขยาย'},
 'hh_id': ObjectId('647e3af1183724b80f000001'),
 'province_code': '10'}

mso_member | top-level fields: 60
{'_id': ObjectId('647e3afb40869f1d22000003'),
 'birthdate': datetime.datetime(1970, 6, 11, 0, 0),
 'fam_id': ObjectId('647e3af2ce02a3d3e2000002'),
 'hh_id': ObjectId('647e3af1183724b80f000001'),
 'prob_summary': {'prob_occupation': [{'code': 'PBOC6',
                                       'desc': 'มีหนี้สิน',
                                       'plans': [{'activities': [],
                                                  'main_org_code': '03000',
                                                  'main_org_name': 'กระทรวงการคลัง',
            

`mso_household` เก็บหนึ่งที่อยู่อาศัยและเชื่อมกับ `mso_family` ผ่าน `family.hh_id → household._id`
ส่วน `mso_member` อ้างทั้ง `hh_id` และ `fam_id` จึงระบุตัวบุคคล ครอบครัว และครัวเรือนได้
สมาชิกฝังสวัสดิการ ปัญหา และแผนไว้ในเอกสารเพราะมักอ่านและปรับปรุงพร้อมข้อมูลรายบุคคล
ข้อดีเมื่อเทียบ relational คือดึงประวัติและ array ที่เกี่ยวข้องได้ในครั้งเดียวโดยลด join
ข้อเสียคือโค้ดปัญหากระจายหลาย array และมีข้อมูลซ้ำระหว่างข้อเท็จจริงกับแผน จึงรวมผลและรักษาความสอดคล้องได้ยาก

### O1. สมาชิกที่ยังมีชีวิตรายจังหวัด

**คำถาม:** จำนวนสมาชิกในแต่ละจังหวัดเป็นเท่าไร?

In [3]:
o1 = [{'$match': ACTIVE}, {'$group': {'_id': {'code': '$province_code', 'name': '$province_name'}, 'members': {'$sum': 1}}}, {'$sort': {'members': -1}}]
show('O1', o1)
pass

,members,_id.code,_id.name
0,1353,10,กรุงเทพมหานคร
1,1211,40,ขอนแก่น
2,1117,30,นครราชสีมา
3,1013,50,เชียงใหม่
4,975,14,พระนครศรีอยุธยา
5,908,94,ปัตตานี
6,691,58,แม่ฮ่องสอน


**ตีความ:** จำนวนสมาชิกต่างกันระหว่างจังหวัด สะท้อนองค์ประกอบชุดข้อมูล จึงต้องเทียบอัตราภายในแต่ละจังหวัดเมื่อวิเคราะห์ปัญหา

### O2. ครัวเรือนแต่ละประเภท

**คำถาม:** ครัวเรือนที่มีสมาชิกยังมีชีวิตมีกี่หลังในแต่ละประเภท?

In [4]:
o2 = [{'$match': ACTIVE}] + HH_JOIN + [
    {'$group': {'_id': '$hh_id', 'hh_type': {'$first': {'$ifNull': ['$hh.hh_type.code', 'UNKNOWN']}}}},
    {'$group': {'_id': '$hh_type', 'households': {'$sum': 1}}}, {'$sort': {'_id': 1}}
]
show('O2', o2)
pass

,_id,households
0,HHT1,351
1,HHT2,854
2,HHT3,747


**ตีความ:** นับ hh_id ไม่ซ้ำก่อนรวมประเภท จึงเป็นจำนวนครัวเรือนที่มีสมาชิกมีชีวิต ไม่ใช่จำนวนสมาชิกที่อาศัยในครัวเรือนประเภทนั้น

### O3. ช่วงอายุ

**คำถาม:** สมาชิกอยู่ในช่วงอายุใดบ้าง?

In [5]:
o3 = [{'$match': ACTIVE}, {'$set': {'age': AGE}}, {'$bucket': {'groupBy': '$age', 'boundaries': [0, 15, 30, 45, 60, 70, 80, 120], 'default': 'unknown/outside', 'output': {'members': {'$sum': 1}}}}]
show('O3', o3)
pass

,_id,members
0,0,2347
1,15,852
2,30,1339
3,45,1269
4,60,548
5,70,521
6,80,392


**ตีความ:** กลุ่มอายุ 60 ปีขึ้นไปเป็นฐานสำหรับวิเคราะห์ช่องว่างเบี้ย โดยช่วง bucket รวมขอบล่างและไม่รวมขอบบน

### O4. ปัญหาที่พบบ่อย

**คำถาม:** สิบโค้ดปัญหาที่บันทึกบ่อยที่สุดคืออะไร?

In [6]:
PROB_FIELDS = ['prob_accessibility', 'prob_education', 'prob_family', 'prob_health', 'prob_housing', 'prob_occupation', 'prob_violence', 'prob_human_trafficking', 'prob_others']
o4 = [{'$match': ACTIVE}, {'$project': {'codes': {'$setUnion': [arr(p + '.code') for p in PROB_FIELDS]}}},
      {'$unwind': '$codes'}, {'$group': {'_id': '$codes', 'members': {'$sum': 1}}}, {'$sort': {'members': -1, '_id': 1}}, {'$limit': 10}]
show('O4', o4)
pass

,_id,members
0,PBOC3,1857
1,PBOC6,1416
2,PBOC7,1228
3,PBHE6,788
4,PBOC14,673
5,PBOC8,599
6,PBHO1,549
7,PBFA21,518
8,PBHO4,409
9,PBFA18,349


**ตีความ:** ใช้ setUnion ให้แต่ละโค้ดนับหนึ่งครั้งต่อคน โค้ดที่บันทึกบ่อยจึงบอกภาระเคสที่เห็นในระบบ แต่ไม่ครอบคลุมปัญหาที่ไม่ได้บันทึก

### O5. สถานะปัญหาทั้งระบบ

**คำถาม:** รายการปัญหาใน prob_summary มีแผนและติดตามแล้วเพียงใด?

In [7]:
SUMMARY_ARRAY = {'$reduce': {'input': {'$objectToArray': {'$ifNull': ['$prob_summary', {}]}}, 'initialValue': [], 'in': {'$concatArrays': ['$$value', '$$this.v']}}}
o5 = [{'$match': ACTIVE}, {'$project': {'cases': SUMMARY_ARRAY}}, {'$unwind': '$cases'},
      {'$group': {'_id': '$cases.status', 'problem_entries': {'$sum': 1}}}, {'$sort': {'_id': 1}}]
overview_status = show('O5', o5)
display(Markdown(f"สถานะ 0 ไม่มีแผน: **{overview_status.loc[overview_status['_id'].eq(0), 'problem_entries'].sum():,} / {overview_status.problem_entries.sum():,} รายการ**"))
pass

,_id,problem_entries
0,0,3519
1,1,2452
2,2,3174
3,3,1461


สถานะ 0 ไม่มีแผน: **3,519 / 10,606 รายการ**

**ตีความ:** หน่วยนับเป็นรายการปัญหา คนหนึ่งมีหลายรายการได้ สถานะ 0 เป็นงานที่ยังไม่มีแผนและไม่ควรนำจำนวนนี้ไปอ้างเป็นจำนวนคน

## Task 2 — State the problem
ข้อความภาษาอังกฤษด้านล่างมี 150–250 คำตามข้อกำหนด และเติมตัวเลขจากผลจริงเมื่อจบการรัน

In [8]:
baseline = agg('mso_member', ELDERS + rate_group(None)).iloc[0]
alternative = agg('mso_member', [{'$match': {**ACTIVE, 'is_disabled': 1}}, {'$group': {'_id': None, 'disabled': {'$sum': 1}, 'no_card': {'$sum': {'$cond': [{'$eq': ['$have_pwd_card', 0]}, 1, 0]}}}}]).iloc[0]
statement = f"""The priority problem is a possible old-age allowance access or recording gap among living, non-removed members aged 60 or over at the assignment's 30 June 2024 reference snapshot. Operationally, the group has no WFEL9 code in welf_eldery. It contains {int(baseline.gap):,} of {int(baseline.elderly):,} older members ({baseline.gap_pct:.1f}%). Missing welfare records do not establish legal eligibility or prove that payments were never received.

For fragile households, unreliable income can constrain food, transport and medicine, especially when chronic illness or monthly debt repayments coexist. The analysis therefore compares housing registration and living arrangements, and profiles economic resources, health and other welfare. These associations guide verification priorities rather than establish causes.

Action is timely at the next case-review cycle because uncoded gaps may otherwise remain outside officers' plans. The proposed response is proactive verification and registration support, with referrals where appropriate, followed by a six-month review. An alternative considered was disabled members without a disability card: {int(alternative.no_card)} of {int(alternative.disabled)} ({100*alternative.no_card/alternative.disabled:.1f}%). Its higher rate warrants attention, but the elderly problem covers more people in this dataset and allows an integrated housing, health and welfare response. This is a reasoned scope choice, not evidence that disability needs matter less. All MSO counts are synthetic and cannot estimate Thailand's actual unmet need."""
assert 150 <= len(statement.split()) <= 250
display(Markdown(statement))
print('Word count (whitespace, English):', len(statement.split()))
display(alternative.to_frame('value'))
pass

The priority problem is a possible old-age allowance access or recording gap among living, non-removed members aged 60 or over at the assignment's 30 June 2024 reference snapshot. Operationally, the group has no WFEL9 code in welf_eldery. It contains 243 of 1,461 older members (16.6%). Missing welfare records do not establish legal eligibility or prove that payments were never received.

For fragile households, unreliable income can constrain food, transport and medicine, especially when chronic illness or monthly debt repayments coexist. The analysis therefore compares housing registration and living arrangements, and profiles economic resources, health and other welfare. These associations guide verification priorities rather than establish causes.

Action is timely at the next case-review cycle because uncoded gaps may otherwise remain outside officers' plans. The proposed response is proactive verification and registration support, with referrals where appropriate, followed by a six-month review. An alternative considered was disabled members without a disability card: 158 of 374 (42.2%). Its higher rate warrants attention, but the elderly problem covers more people in this dataset and allows an integrated housing, health and welfare response. This is a reasoned scope choice, not evidence that disability needs matter less. All MSO counts are synthetic and cannot estimate Thailand's actual unmet need.

Word count (whitespace, English): 204


,value
_id,None
disabled,374
no_card,158


## Task 3 — Evidence from the data

### Q1. ขนาดปัญหาและตำแหน่งที่ตั้ง

**คำถาม:** จำนวนและอัตราไม่พบ WFEL9 แตกต่างกันตามจังหวัดและอำเภออย่างไร?

In [9]:
q1p = ELDERS + rate_group({'code': '$province_code', 'name': '$province_name'})
q1d = ELDERS + rate_group({'province': '$province_code', 'district': '$amphur_code', 'name': '$amphur_name'})
province_rates = show('Q1_province', q1p)
district_rates = show('Q1_district', q1d)
display(Markdown(f"รวม **{int(baseline.gap):,}/{int(baseline.elderly):,} คน ({baseline.gap_pct:.2f}%)**; อำเภออันดับแรกตามอัตรา: {district_rates.iloc[0]['_id']}"))
pass

,elderly,gap,gap_pct,_id.code,_id.name
0,135,43,31.85,58,แม่ฮ่องสอน
1,199,43,21.61,50,เชียงใหม่
2,224,39,17.41,30,นครราชสีมา
3,234,34,14.53,40,ขอนแก่น
4,187,26,13.90,14,พระนครศรีอยุธยา
5,286,39,13.64,10,กรุงเทพมหานคร
6,196,19,9.69,94,ปัตตานี


,elderly,gap,gap_pct,_id.province,_id.district,_id.name
0,70,27,38.57,58,5801,เมืองแม่ฮ่องสอน
1,65,16,24.62,58,5803,ปาย
2,95,22,23.16,50,5003,แม่แจ่ม
3,104,21,20.19,50,5001,เมืองเชียงใหม่
4,134,26,19.40,30,3001,เมืองนครราชสีมา
5,110,20,18.18,10,1033,คลองเตย
6,136,21,15.44,40,4001,เมืองขอนแก่น
7,90,13,14.44,30,3002,ครบุรี
8,86,12,13.95,14,1402,ท่าเรือ
9,101,14,13.86,14,1401,พระนครศรีอยุธยา


รวม **243/1,461 คน (16.63%)**; อำเภออันดับแรกตามอัตรา: {'province': '58', 'district': '5801', 'name': 'เมืองแม่ฮ่องสอน'}

**ตีความ:** ใช้ผู้สูงอายุทั้งหมดในพื้นที่เป็นตัวหาร ไม่ใช่ประชากรทุกวัย; พื้นที่ฐานเล็กอาจมีอัตราสูงโดยจำนวนคนไม่มาก จึงต้องอ่านทั้งจำนวนและอัตรา

### Q2. เปรียบเทียบที่อยู่อาศัย

**คำถาม:** การไม่พบเบี้ยสัมพันธ์กับบ้านไม่มีเลขที่และการอยู่คนเดียวหรือไม่?

In [10]:
q2 = ELDERS + HH_JOIN + [{'$facet': {
    'house_number': rate_group({'$ifNull': ['$hh.have_num', 'UNKNOWN']}),
    'household_type': rate_group({'$ifNull': ['$hh.hh_type.code', 'UNKNOWN']})
}}]
housing = list(db.mso_member.aggregate(q2))[0]
for label, records in housing.items():
    RESULTS['Q2_' + label] = pd.DataFrame(records)
    print(label); display(RESULTS['Q2_' + label])
join_audit = agg('mso_member', ELDERS + HH_JOIN + [{'$group': {'_id': None, 'retained': {'$sum': 1}, 'unmatched': {'$sum': {'$cond': [{'$eq': [{'$type': '$hh._id'}, 'missing']}, 1, 0]}}}}])
display(join_audit)
print('Housing comparison base:', int(join_audit.retained.iloc[0]), 'vs all elders:', int(baseline.elderly), '(removed household records excluded).')
print('No number: 37/81 = 45.68%; numbered: 205/1,378 = 14.88%; living alone: 63/245 = 25.71%.')
pass

house_number


,_id,elderly,gap,gap_pct
0,0,81,37,45.68
1,1,1378,205,14.88


household_type


,_id,elderly,gap,gap_pct
0,HHT1,245,63,25.71
1,HHT3,1010,151,14.95
2,HHT2,204,28,13.73


,_id,retained,unmatched
0,None,1459,0


Housing comparison base: 1459 vs all elders: 1461 (removed household records excluded).
No number: 37/81 = 45.68%; numbered: 205/1,378 = 14.88%; living alone: 63/245 = 25.71%.


**ตีความ:** เทียบอัตราภายในกลุ่ม house number และ hh_type โดยแยก UNKNOWN; ความต่างอาจบอกอุปสรรคการเข้าถึง แต่ข้อมูลนี้ยังไม่พิสูจน์ว่าการไม่มีเลขที่เป็นสาเหตุ

### Q3a. เศรษฐกิจ

**คำถาม:** รายได้ การทำงาน เงินออม และภาระชำระหนี้ของกลุ่มเป้าหมายต่างจากผู้มีบันทึกเบี้ยอย่างไร?

In [11]:
RECOMPUTED_INCOME = {'$add': [{'$ifNull': ['$pri_occ_income', 0]}, {'$ifNull': ['$sec_occ_income', 0]}, {'$ifNull': ['$subsidy_amount', 0]}]}
q3a = ELDERS + [
 {'$set': {'income_recomputed': RECOMPUTED_INCOME, 'income_known': {'$gt': [{'$size': {'$filter': {'input': [{'$type': '$pri_occ_income'}, {'$type': '$sec_occ_income'}, {'$type': '$subsidy_amount'}], 'as': 't', 'cond': {'$in': ['$$t', ['int', 'long', 'double', 'decimal']]}}}}, 0]}, 'repayment': {'$sum': arr('debt.amount')}, 'saving_total': {'$sum': arr('saving.amount')}}},
 {'$group': {'_id': '$missing_allowance', 'n': {'$sum': 1},
     'working': {'$sum': {'$cond': [{'$eq': ['$occup_status', 1]}, 1, 0]}},
     'primary_income_mean': {'$avg': '$pri_occ_income'}, 'secondary_income_mean': {'$avg': '$sec_occ_income'}, 'support_mean': {'$avg': '$subsidy_amount'},
     'income_known_n': {'$sum': {'$cond': ['$income_known', 1, 0]}},
     'recomputed_income_mean': {'$avg': {'$cond': ['$income_known', '$income_recomputed', None]}},
     'reported_income_mean': {'$avg': '$SUM_INCOME'}, 'spending_mean': {'$avg': '$SUM_EXPENDITURE'},
     'have_debt_n': {'$sum': {'$cond': [{'$eq': ['$have_debt', 1]}, 1, 0]}},
     'informal_debt_n': {'$sum': {'$cond': [has('debt.type', 'นอกระบบ'), 1, 0]}},
     'repayment_mean': {'$avg': '$repayment'}, 'saving_mean': {'$avg': '$saving_total'},
     'income_mismatch_n': {'$sum': {'$cond': [{'$and': ['$income_known', {'$ne': ['$SUM_INCOME', '$income_recomputed']}]}, 1, 0]}}
 }}, {'$sort': {'_id': 1}}
]
show('Q3a_economy', q3a)
print('Income detail and reported totals are presented separately; compare True (gap) with False (recorded allowance), using income_known_n for the recomputed mean.')
print('Reported mean income is 3,560 baht in the gap group vs 2,606 baht in the recorded group: the gap group is not uniformly poorer. Target verification by access barriers and individual need rather than assuming no income.')
pass

_id,False,True
n,"1,218.00",243.00
working,360.00,100.00
primary_income_mean,"6,741.67","6,911.00"
secondary_income_mean,"1,704.00","1,689.47"
support_mean,"1,578.74","1,696.97"
income_known_n,741.00,166.00
recomputed_income_mean,"4,259.51","5,031.33"
reported_income_mean,"2,606.46","3,560.08"
spending_mean,"4,256.16","4,238.68"
have_debt_n,523.00,110.00


Income detail and reported totals are presented separately; compare True (gap) with False (recorded allowance), using income_known_n for the recomputed mean.
Reported mean income is 3,560 baht in the gap group vs 2,606 baht in the recorded group: the gap group is not uniformly poorer. Target verification by access barriers and individual need rather than assuming no income.


**ตีความ:** แถว True คือไม่พบเบี้ย; ค่าเฉลี่ย detail ใช้เฉพาะค่าที่มีบันทึก ส่วนผลรวม array ว่างสมมติเป็นศูนย์และแสดงจำนวน income_known กำกับ จึงไม่ควรอ้างว่าไม่มีรายได้จริง; debt.amount เป็นภาระชำระรายเดือน ไม่ใช่ยอดหนี้คงค้าง และรายได้ที่คำนวณใหม่ไม่รวมมูลค่าเบี้ยที่ไม่มีจำนวนเงิน

### Q3b. สุขภาพและความพิการ

**คำถาม:** กลุ่มไม่พบเบี้ยมีโรคเรื้อรังและปัญหาสิทธิรักษาหรือไม่?

In [12]:
q3b = ELDERS + [{'$group': {'_id': '$missing_allowance', 'n': {'$sum': 1},
 'chronic_n': {'$sum': {'$cond': [{'$gt': [{'$size': arr('chronic_disease.code')}, 0]}, 1, 0]}},
 'hypertension_n': {'$sum': {'$cond': [has('chronic_disease.code', 'CD2'), 1, 0]}},
 'no_medical_coverage_n': {'$sum': {'$cond': [has('med_scheme.code', 'MS7'), 1, 0]}},
 'medical_record_missing_n': {'$sum': {'$cond': [{'$eq': [{'$size': arr('med_scheme')}, 0]}, 1, 0]}},
 'disabled_n': {'$sum': {'$cond': [{'$eq': ['$is_disabled', 1]}, 1, 0]}},
 'disabled_without_card_n': {'$sum': {'$cond': [{'$and': [{'$eq': ['$is_disabled', 1]}, {'$eq': ['$have_pwd_card', 0]}]}, 1, 0]}}
}}, {'$set': {'chronic_pct': pct('$chronic_n', '$n')}}, {'$sort': {'_id': 1}}]
show('Q3b_health', q3b)
print('Chronic conditions: gap 144/243 (59.26%) vs recorded allowance 673/1,218 (55.25%); explicit MS7: 16/243 versus 24/1,218.')
pass

,_id,n,chronic_n,hypertension_n,no_medical_coverage_n,medical_record_missing_n,disabled_n,disabled_without_card_n,chronic_pct
0,False,1218,673,336,24,0,159,78,55.25
1,True,243,144,70,16,0,32,17,59.26


Chronic conditions: gap 144/243 (59.26%) vs recorded allowance 673/1,218 (55.25%); explicit MS7: 16/243 versus 24/1,218.


**ตีความ:** โรคเรื้อรังและความพิการที่พบร่วมกับการไม่พบบันทึกเบี้ยสนับสนุนการประสานบริการหลายด้าน; MS7 หมายถึงไม่มีสิทธิรักษาตามบันทึก ส่วน array หายเป็นข้อมูลไม่ครบ จึงรายงานแยกกัน

### Q3c. สวัสดิการอื่นและสัญชาติ

**คำถาม:** ผู้ไม่พบเบี้ยมีสวัสดิการอื่นรองรับหรือไม่ และองค์ประกอบสัญชาติเป็นอย่างไร?

In [13]:
q3c = ELDERS + [{'$group': {'_id': '$missing_allowance', 'n': {'$sum': 1},
 'welfare_card_n': {'$sum': {'$cond': [has('welf_others.code', 'WFOT4'), 1, 0]}},
 'disability_allowance_n': {'$sum': {'$cond': [has('welf_disability.code', 'WFDA14'), 1, 0]}},
 'hardship_grant_n': {'$sum': {'$cond': [has('welf_eldery.code', 'WFEL2'), 1, 0]}}
}}, {'$sort': {'_id': 1}}]
show('Q3c_welfare', q3c)
show('Q3c_nationality', ELDERS + rate_group({'$ifNull': ['$nationality', 'UNKNOWN']}))
print('Among 243 gap cases, 140 have WFOT4, 13 have WFDA14 and 21 have WFEL2; these supports do not substitute for verifying WFEL9.')
pass

,_id,n,welfare_card_n,disability_allowance_n,hardship_grant_n
0,False,1218,738,74,105
1,True,243,140,13,21


,_id,elderly,gap,gap_pct
0,อื่นๆ,15,14,93.33
1,ไทย,1446,229,15.84


Among 243 gap cases, 140 have WFOT4, 13 have WFDA14 and 21 have WFEL2; these supports do not substitute for verifying WFEL9.


**ตีความ:** บัตรสวัสดิการ เงินช่วยเหลือ และเบี้ยพิการเป็นคนละรายการกับ WFEL9; สัญชาติช่วยเตือนว่าบางเคสต้องตรวจช่องทางช่วยเหลืออื่นและเงื่อนไขสิทธิ ไม่ควรสรุปว่าทุกคนที่ไม่พบ WFEL9 ต้องได้รับเบี้ย

### Q4a. สถานะการตอบสนอง

**คำถาม:** ในกลุ่มไม่พบเบี้ย มี PBAC8 ที่มีแผนแล้วกี่คน?

In [14]:
q4a = TARGET + [
 {'$unwind': '$prob_summary.prob_accessibility'},
 {'$match': {'prob_summary.prob_accessibility.code': 'PBAC8'}},
 {'$group': {'_id': '$prob_summary.prob_accessibility.status', 'entries': {'$sum': 1}, 'ids': {'$addToSet': '$_id'}}},
 {'$project': {'entries': 1, 'people': {'$size': '$ids'}}}, {'$sort': {'_id': 1}}
]
show('Q4a_status', q4a)
pass

,_id,entries,people
0,0,35,35
1,1,19,19
2,2,25,25
3,3,15,15


**ตีความ:** สถานะ 0 ไม่มีแผน 1 มีแผน 2 ติดตามแล้ว และ 3 ปิดเคส; นับเฉพาะ PBAC8 ของกลุ่มเป้าหมาย เพราะ PBAC8 ครอบคลุมทั้งเบี้ยสูงอายุและเบี้ยพิการและไม่ใช่หลักฐานว่ารับเงินจริง

### Q4b. ประเภทแผนและผลติดตาม

**คำถาม:** แผน PBAC8 ใช้แนวทางใด และบันทึกผลติดตามอย่างไร?

In [15]:
q4b = TARGET + [
 {'$unwind': '$prob_summary.prob_accessibility'}, {'$match': {'prob_summary.prob_accessibility.code': 'PBAC8'}},
 {'$unwind': {'path': '$prob_summary.prob_accessibility.plans', 'preserveNullAndEmptyArrays': True}},
 {'$unwind': {'path': '$prob_summary.prob_accessibility.plans.activities', 'preserveNullAndEmptyArrays': True}},
 {'$group': {'_id': {
   'solution_type': {'$ifNull': ['$prob_summary.prob_accessibility.plans.sol_type', 'NO_PLAN']},
   'solution_code': {'$ifNull': ['$prob_summary.prob_accessibility.plans.sol_code', 'NO_PLAN']},
   'solution_desc': {'$ifNull': ['$prob_summary.prob_accessibility.plans.sol_des', 'NO_PLAN']},
   'plan_status': {'$ifNull': ['$prob_summary.prob_accessibility.plans.status', 'NO_PLAN']},
   'progress': {'$ifNull': ['$prob_summary.prob_accessibility.plans.activities.act_progression', 'NO_FOLLOWUP']}
 }, 'rows': {'$sum': 1}, 'ids': {'$addToSet': '$_id'}}},
 {'$project': {'rows': 1, 'people': {'$size': '$ids'}}}, {'$sort': {'people': -1}}
]
show('Q4b_plans', q4b)
pass

,rows,people,_id.solution_type,_id.solution_code,_id.solution_desc,_id.plan_status,_id.progress
0,35,35,NO_PLAN,NO_PLAN,NO_PLAN,NO_PLAN,NO_FOLLOWUP
1,14,14,ประสานส่งต่อ,TR47,เบี้ยยังชีพผู้สูงอายุ,0,NO_FOLLOWUP
2,15,13,ประสานส่งต่อ,TR44,เบี้ยยังชีพคนพิการ,1,อยู่รอด
3,11,11,ประสานส่งต่อ,TR44,เบี้ยยังชีพคนพิการ,0,NO_FOLLOWUP
4,9,9,ประสานส่งต่อ,TR47,เบี้ยยังชีพผู้สูงอายุ,1,พอเพียง
5,8,8,ประสานส่งต่อ,TR44,เบี้ยยังชีพคนพิการ,1,พอเพียง
6,11,7,ประสานส่งต่อ,TR47,เบี้ยยังชีพผู้สูงอายุ,1,อยู่รอด
7,8,7,ประสานส่งต่อ,TR44,เบี้ยยังชีพคนพิการ,1,ยั่งยืน
8,7,6,ประสานส่งต่อ,TR44,เบี้ยยังชีพคนพิการ,2,พอเพียง
9,8,6,ประสานส่งต่อ,TR47,เบี้ยยังชีพผู้สูงอายุ,2,พอเพียง


**ตีความ:** หนึ่งคนมีหลายแผนและกิจกรรมได้ people จึงนับไม่ซ้ำภายในแถว แต่ห้ามบวกข้ามแถวเป็นจำนวนคนทั้งหมด; NO_FOLLOWUP คือไม่มีบันทึกผล ไม่ได้แปลว่าการช่วยเหลือล้มเหลว

### Q5. ช่องว่างระหว่างข้อเท็จจริงกับการจัดการเคส

**คำถาม:** ผู้ไม่พบ WFEL9 มี PBAC8 ในแบบบันทึกปัญหาและในแผนสอดคล้องกันหรือไม่?

In [16]:
q5 = TARGET + [
 {'$set': {'has_problem_code': has('prob_accessibility.code', 'PBAC8'), 'has_case_code': has('prob_summary.prob_accessibility.code', 'PBAC8')}},
 {'$group': {'_id': {'problem_array': '$has_problem_code', 'case_management': '$has_case_code'}, 'people': {'$sum': 1}}}, {'$sort': {'people': -1}}
]
gap_recording = show('Q5_gap', q5)
pass

,people,_id.problem_array,_id.case_management
0,149,False,False
1,94,True,True


**ตีความ:** เคสที่ไม่พบทั้ง WFEL9 และ PBAC8 เป็นลำดับงานตรวจสอบเชิงรุกที่ค้นจากข้อเท็จจริงได้ แม้ระบบยังไม่ได้เปิดประเด็นปัญหา; absence ของโค้ดอาจเป็นช่องว่างการบันทึก

### Q5s. ตรวจความไวต่อนิยามอายุ

**คำถาม:** ถ้าใช้อายุครบวันเกิดแทนการนับขอบเขตปี ผลรวมเปลี่ยนเท่าไร?

In [17]:
q5s = [{'$match': ACTIVE}, {'$set': FLAGS}, {'$facet': {
 'assignment_year_boundaries': [{'$match': {'age': {'$gte': 60}}}] + rate_group(None),
 'completed_birthday': [{'$match': {'birthdate': {'$lte': datetime(1964, 6, 30)}}}] + rate_group(None)
}}]
age_sensitivity = list(db.mso_member.aggregate(q5s))[0]
for label, records in age_sensitivity.items(): print(label); display(pd.DataFrame(records))
pass

assignment_year_boundaries


,_id,elderly,gap,gap_pct
0,None,1461,243,16.63


completed_birthday


,_id,elderly,gap,gap_pct
0,None,1440,223,15.49


**ตีความ:** รายงานหลักใช้สูตรที่โจทย์กำหนดเพื่อให้ตรวจซ้ำได้ แต่การปฏิบัติงานจริงต้องใช้อายุครบวันเกิดและวันตรวจสอบปัจจุบัน; ตารางนี้เปิดเผยผลของทางเลือกดังกล่าว

## Task 4 — Combine external data
**ผู้เผยแพร่:** สำนักงานสถิติแห่งชาติ (NSO), ข้อมูลต้นทางกรมการปกครอง (DOPA)  
**ชุดข้อมูล:** ประชากรอายุ 60 ปีขึ้นไป จากการทะเบียน จำแนกเพศและจังหวัด, OS_0405_01_0008_01  
**Catalog:** https://gdcatalog.go.th/dataset/gdpublish-605  
**Download:** https://catalogapi.nso.go.th/api/index?table=OS_0405_01_0008_01&format=json  
**วันที่เข้าถึง:** 28 ตุลาคม 2026 | **ปีที่ใช้ join:** 2024 (พ.ศ. 2567) | **ปีล่าสุดในไฟล์:** 2025 (พ.ศ. 2568)

โหลดต้นฉบับและสคริปต์เตรียมข้อมูลไว้ใน `ext_data/`; ใช้เฉพาะเพศ “รวม” ลบคำว่า “จังหวัด” และจับคู่ชื่อผ่าน provinces.csv
เลือกปี 2024 ให้ตรงปี snapshot แต่ข้อมูลรายปียังต่างจาก 30 มิถุนายน จึงไม่ถือว่าช่วงเวลาตรงกันทุกวัน
เก็บปีล่าสุดประกอบบริบทด้วย และไม่ใช้จำนวนตัวอย่างสังเคราะห์แทนความครอบคลุมจริง

In [18]:
external = pd.read_csv(ROOT / 'ext_data/province_elderly.csv', dtype={'province_code': str})
assert external.province_code.str.fullmatch(r'\d{2}').all()
assert not external.province_code.duplicated().any()
records = json.loads(external.to_json(orient='records', force_ascii=False))
for record in records:
    db.ext_province_elderly.replace_one({'province_code': record['province_code']}, record, upsert=True)
db.ext_province_elderly.create_index('province_code', unique=True)
display(external[['province_code', 'province_name_th', 'elderly_2024', 'elderly_latest', 'latest_year']])
print('Loaded ext_province_elderly:', db.ext_province_elderly.count_documents({}))
pass

,province_code,province_name_th,elderly_2024,elderly_latest,latest_year
0,10,กรุงเทพมหานคร,1264604,1305293,2025
1,14,พระนครศรีอยุธยา,179184,186511,2025
2,30,นครราชสีมา,551419,574051,2025
3,40,ขอนแก่น,378450,394314,2025
4,50,เชียงใหม่,406871,419480,2025
5,58,แม่ฮ่องสอน,40031,41623,2025
6,94,ปัตตานี,99590,102536,2025


Loaded ext_province_elderly: 7


### Q6. ประชากรจริงช่วยเพิ่มบริบทอย่างไร

**คำถาม:** จังหวัดที่มีอัตราช่องว่างสูงในชุดสังเคราะห์มีประชากรผู้สูงอายุจริงเท่าไร?

In [19]:
q6 = ELDERS + rate_group('$province_code') + [
 {'$lookup': {'from': 'ext_province_elderly', 'localField': '_id', 'foreignField': 'province_code', 'as': 'ext'}},
 {'$unwind': {'path': '$ext', 'preserveNullAndEmptyArrays': True}},
 {'$project': {'_id': 0, 'province_code': '$_id', 'province': '$ext.province_name_th', 'synthetic_elderly': '$elderly', 'synthetic_gap': '$gap', 'synthetic_gap_pct': '$gap_pct', 'registered_elderly_2024': '$ext.elderly_2024', 'registered_elderly_latest': '$ext.elderly_latest', 'latest_year': '$ext.latest_year', 'registered_growth_pct': pct({'$subtract': ['$ext.elderly_latest', '$ext.elderly_2024']}, '$ext.elderly_2024')}},
 {'$sort': {'synthetic_gap_pct': -1}}
]
external_context = show('Q6_external', q6)
assert len(external_context) == 7 and external_context.registered_elderly_2024.notna().all()
display(Markdown('**การตรวจ join:** ครบ 7/7 จังหวัด ไม่มีรหัสตกหล่น'))
pass

,province_code,province,synthetic_elderly,synthetic_gap,synthetic_gap_pct,registered_elderly_2024,registered_elderly_latest,latest_year,registered_growth_pct
0,58,แม่ฮ่องสอน,135,43,31.85,40031,41623,2025,3.98
1,50,เชียงใหม่,199,43,21.61,406871,419480,2025,3.10
2,30,นครราชสีมา,224,39,17.41,551419,574051,2025,4.10
3,40,ขอนแก่น,234,34,14.53,378450,394314,2025,4.19
4,14,พระนครศรีอยุธยา,187,26,13.90,179184,186511,2025,4.09
5,10,กรุงเทพมหานคร,286,39,13.64,1264604,1305293,2025,3.22
6,94,ปัตตานี,196,19,9.69,99590,102536,2025,2.96


**การตรวจ join:** ครบ 7/7 จังหวัด ไม่มีรหัสตกหล่น

**ตีความ:** ข้อมูลภายนอกแสดงขนาดประชากรผู้สูงอายุจริงซึ่งฐานสังเคราะห์บอกไม่ได้: จังหวัดที่มีอัตราช่องว่างสูงไม่จำเป็นต้องมีประชากรมาก จึงเลือกพื้นที่นำร่องจากอัตราภายในตัวอย่างได้ แต่การขยายกำลังคนต้องสำรวจฐานจริงก่อน ห้ามนำ synthetic_gap_pct ไปคูณประชากรจริงเพื่ออ้างจำนวนตกหล่นทั้งจังหวัด

### Field log — ฟิลด์ที่ใช้เป็นหลักฐานจริง
ไม่นับ status และ vital_status ที่ใช้คัดออก และไม่ใช้ฟิลด์ที่แค่ project เป็นหลักฐาน

In [20]:
field_rows = [
 ('mso_member','birthdate','Q1, Q5s','Defines age base; sensitivity changes count'),
 ('mso_member','province_code','Q1, Q6','Province-specific numerator and denominator; external join'),
 ('mso_member','amphur_code','Q1','District-specific rate'),
 ('mso_member','welf_eldery.code','Q1–Q6','WFEL9 gap; WFEL2 alternative support'),
 ('mso_household','have_num','Q2','Compare numberless and numbered homes'),
 ('mso_household','hh_type.code','Q2','Compare living alone and family households'),
 ('mso_member','occup_status','Q3a','Share currently working'),
 ('mso_member','pri_occ_income','Q3a','Primary-income mean and recomputed resources'),
 ('mso_member','sec_occ_income','Q3a','Secondary-income mean and resources'),
 ('mso_member','subsidy_amount','Q3a','Support amount mean and resources'),
 ('mso_member','SUM_INCOME','Q3a','Compare reported total and recomputed income'),
 ('mso_member','SUM_EXPENDITURE','Q3a','Spending pressure'),
 ('mso_member','have_debt','Q3a','Number with recorded debt'),
 ('mso_member','debt.type','Q3a','Informal borrowing exposure'),
 ('mso_member','debt.amount','Q3a','Monthly repayment burden, not debt stock'),
 ('mso_member','saving.amount','Q3a','Recorded savings buffer'),
 ('mso_member','chronic_disease.code','Q3b','Chronic illness and hypertension'),
 ('mso_member','med_scheme.code','Q3b','Explicit no-coverage code MS7'),
 ('mso_member','is_disabled','Q3b','Disability co-occurrence'),
 ('mso_member','have_pwd_card','Q3b','Disability-card gap conditional on disability'),
 ('mso_member','welf_others.code','Q3c','State welfare card as other support'),
 ('mso_member','welf_disability.code','Q3c','Disability allowance co-occurrence'),
 ('mso_member','nationality','Q3c','Separates verification needs by recorded nationality'),
 ('mso_member','prob_summary.prob_accessibility.code','Q4, Q5','PBAC8 case-management recognition'),
 ('mso_member','prob_summary.prob_accessibility.status','Q4a','Problem response status'),
 ('mso_member','prob_summary.prob_accessibility.plans.sol_type','Q4b','Plan type'),
 ('mso_member','prob_summary.prob_accessibility.plans.sol_code','Q4b','Referral route including TR47'),
 ('mso_member','prob_summary.prob_accessibility.plans.status','Q4b','Plan status'),
 ('mso_member','prob_summary.prob_accessibility.plans.activities.act_progression','Q4b','Recorded follow-up outcome'),
 ('mso_member','prob_accessibility.code','Q5','Problem recognition vs case management')
]
field_log = pd.DataFrame(field_rows, columns=['Collection','Field path','Used in query','Why it matters'])
field_log.insert(0, '#', range(1,len(field_log)+1))
display(field_log)
assert len(field_log) >= 15 and field_log.Collection.nunique() >= 2
print('30 distinct evidence fields; member sections: identity/location, economy, health, welfare, problems/case management.')
pass

,#,Collection,Field path,Used in query,Why it matters
0,1,mso_member,birthdate,"Q1, Q5s",Defines age base; sensitivity changes count
1,2,mso_member,province_code,"Q1, Q6",Province-specific numerator and denominator; e...
2,3,mso_member,amphur_code,Q1,District-specific rate
3,4,mso_member,welf_eldery.code,Q1–Q6,WFEL9 gap; WFEL2 alternative support
4,5,mso_household,have_num,Q2,Compare numberless and numbered homes
5,6,mso_household,hh_type.code,Q2,Compare living alone and family households
6,7,mso_member,occup_status,Q3a,Share currently working
7,8,mso_member,pri_occ_income,Q3a,Primary-income mean and recomputed resources
8,9,mso_member,sec_occ_income,Q3a,Secondary-income mean and resources
9,10,mso_member,subsidy_amount,Q3a,Support amount mean and resources


30 distinct evidence fields; member sections: identity/location, economy, health, welfare, problems/case management.


## Task 5 — Policy targeting and monitoring (PDF companion)
ข้อเสนอ: พมจ.แม่ฮ่องสอนร่วมกับ อปท. และ อพม. ตรวจสอบสิทธิและบันทึก WFEL9 ของกลุ่มที่คัดได้
ให้ลำดับเยี่ยมบ้านก่อนแก่ผู้ไม่มีเลขที่หรืออยู่คนเดียว เมื่อยืนยันสิทธิให้ส่งต่อแนวทาง TR47 ซึ่งปรากฏในข้อมูลตัวอย่าง
ผู้ไม่เข้าเงื่อนไขรับเบี้ยต้องประสานสวัสดิการที่เหมาะสมแทน ไม่บันทึกเป็นผู้มีสิทธิอัตโนมัติ
query หนึ่งตัวต่อไปนี้คัดรายชื่อบุคคลเพื่อยืนยันสิทธิในจังหวัดนำร่อง; จำนวนครัวเรือนคำนวณจาก hh_id ไม่ซ้ำ

In [21]:
# Exact same filter is used in the policy PDF and Bonus A2.
# The birthdate range is equivalent to the assignment's calendar-year age rule at SNAPSHOT;
# retain $expr explicitly to preserve the assignment definition. No sort requirement.
TARGET_FILTER = {
 'province_code': '58', 'vital_status': 1, 'status': {'$ne': 'removed'},
 'birthdate': {'$lt': datetime(1965, 1, 1)},
 '$expr': {'$gte': [AGE, 60]}, 'welf_eldery.code': {'$ne': 'WFEL9'}
}
pilot_people = list(db.mso_member.find(TARGET_FILTER, {'_id': 1, 'hh_id': 1, 'amphur_code': 1}))
PILOT_IDS = [x['_id'] for x in pilot_people]
pilot_n = len(pilot_people)
pilot_hh = len({x['hh_id'] for x in pilot_people})
print(json_util.dumps(TARGET_FILTER, indent=2, ensure_ascii=False))
display(pd.DataFrame([{'people_to_verify': pilot_n, 'distinct_households': pilot_hh,
  'staff_hours_at_1_hour_per_person': pilot_n, 'verification_budget_at_300_baht_per_person': pilot_n*300}]))
pilot_priority = agg('mso_member', [{'$match': TARGET_FILTER}] + HH_JOIN + [{'$group': {'_id': None, 'no_house_number': {'$sum': {'$cond': [{'$eq': ['$hh.have_num', 0]},1,0]}}, 'living_alone': {'$sum': {'$cond': [{'$eq': ['$hh.hh_type.code','HHT1']},1,0]}}}}])
display(pilot_priority)
# Persist baseline IDs for a fixed-cohort six-month comparison; original records stay unchanged.
(ROOT/'output/pilot_baseline_ids.json').write_text(json_util.dumps({'snapshot': SNAPSHOT, 'ids': PILOT_IDS}, indent=2), encoding='utf-8')
MONITOR_FILTER = {'_id': {'$in': PILOT_IDS}, **ACTIVE, 'welf_eldery.code': {'$ne': 'WFEL9'}}
print('Six-month remaining baseline cases (current snapshot, not a future observation):', db.mso_member.count_documents(MONITOR_FILTER))
assert pilot_n == sum(1 for _ in db.mso_member.aggregate(TARGET + [{'$match': {'province_code':'58'}}]))
pass

{
  "province_code": "58",
  "vital_status": 1,
  "status": {
    "$ne": "removed"
  },
  "birthdate": {
    "$lt": {
      "$date": {
        "$numberLong": "-157766400000"
      }
    }
  },
  "$expr": {
    "$gte": [
      {
        "$dateDiff": {
          "startDate": "$birthdate",
          "endDate": {
            "$date": "2024-06-30T00:00:00Z"
          },
          "unit": "year"
        }
      },
      60
    ]
  },
  "welf_eldery.code": {
    "$ne": "WFEL9"
  }
}


,people_to_verify,distinct_households,staff_hours_at_1_hour_per_person,verification_budget_at_300_baht_per_person
0,43,40,43,12900


,_id,no_house_number,living_alone
0,None,3,19


Six-month remaining baseline cases (current snapshot, not a future observation): 43


**สมมติฐานต้นทุน:** ตรวจสอบ 1 ชั่วโมงและค่าเดินทาง/ติดต่อ 300 บาทต่อคนเป็นตัวเลขสมมติเพื่อวางแผน
ไม่ใช่อัตราราชการ ไม่รวมงบจ่ายเบี้ยและไม่อ้างการยืนยันสิทธิ; การเยี่ยมหลายคนในบ้านเดียวอาจลดต้นทุน
ในหกเดือนใช้ baseline IDs เดิมนับคนยังไม่พบ WFEL9 แยกผู้ตาย/removed ออกจากเหตุออกจาก cohort
เทียบ `prob_summary.prob_accessibility.status`, `plans.sol_code`, `plans.activities.act_progression` ควบคู่ผลยืนยันสิทธิ
เป้าหมายโครงการที่เสนอคือยืนยันข้อมูลอย่างน้อย 90% ภายใน 60 วัน และลดเคสขาดบันทึกที่ตรวจแล้วว่ามีสิทธิอย่างน้อยครึ่งหนึ่งใน 6 เดือน
เป้าหมายนี้เป็นข้อเสนอ ไม่ใช่ผลที่เกิดขึ้นแล้ว; ฐานเดิมไม่มีฟิลด์ยืนยันสิทธิหรือวันที่เริ่มรับเงินจริง จึงต้องเก็บเพิ่มก่อนประเมินการได้รับเงิน

## Bonus A — Make queries fast

### A1. Dashboard ด้วย $facet

**คำถาม:** ใช้ pipeline เดียวแสดงกลุ่มเป้าหมายตามจังหวัด เพศ และช่วงอายุได้อย่างไร?

In [22]:
a1 = TARGET + [{'$facet': {
 'province': [{'$group': {'_id': '$province_code', 'n': {'$sum': 1}}}, {'$sort': {'n': -1}}],
 'sex': [{'$group': {'_id': '$sex', 'n': {'$sum': 1}}}],
 'age_band': [{'$bucket': {'groupBy': '$age', 'boundaries': [60,70,80,90,120], 'default': 'outside', 'output': {'n': {'$sum': 1}}}}]
}}]
dashboard = list(db.mso_member.aggregate(a1))[0]
for label, rows in dashboard.items(): print(label); display(pd.DataFrame(rows))
assert all(sum(x['n'] for x in dashboard[k]) == int(baseline.gap) for k in dashboard)
pass

province


,_id,n
0,50,43
1,58,43
2,30,39
3,10,39
4,40,34
5,14,26
6,94,19


sex


,_id,n
0,ชาย,105
1,หญิง,138


age_band


,_id,n
0,60,114
1,70,68
2,80,58
3,90,3


**ตีความ:** $match อยู่ก่อน $facet เพื่อลดเอกสารก่อนแยกสามสาขาและให้เงื่อนไขที่รองรับใช้ index ต้นทางได้ ส่วน age ที่คำนวณและ missing welfare ไม่ได้รับประโยชน์จาก index ทุกส่วน

### A2. เปรียบเทียบ executionStats ก่อนและหลัง compound index

In [23]:
INDEX_NAME = 'a4_pilot_esr'
# Only remove this assignment-owned index so rerunning reproduces the before state.
if INDEX_NAME in db.mso_member.index_information(): db.mso_member.drop_index(INDEX_NAME)
def explain_find():
    return db.command('explain', {'find': 'mso_member', 'filter': TARGET_FILTER}, verbosity='executionStats')
def stages(node):
    found = []
    if isinstance(node, dict):
        if 'stage' in node: found.append(node['stage'])
        for v in node.values(): found.extend(stages(v))
    elif isinstance(node,list):
        for v in node: found.extend(stages(v))
    return list(dict.fromkeys(found))
def stats(label, exp):
    s = exp['executionStats']
    return {'run': label, 'winning_stages': ' > '.join(stages(exp['queryPlanner']['winningPlan'])), **{k:s[k] for k in ['totalDocsExamined','totalKeysExamined','nReturned','executionTimeMillis']}}
before = explain_find()
db.mso_member.create_index([('province_code',1), ('vital_status',1), ('birthdate',1)], name=INDEX_NAME)
after = explain_find()
index_results = pd.DataFrame([stats('before', before), stats('after', after)])
display(index_results)
assert before['executionStats']['nReturned'] == after['executionStats']['nReturned'] == pilot_n
pass

,run,winning_stages,totalDocsExamined,totalKeysExamined,nReturned,executionTimeMillis
0,before,FETCH > IXSCAN,715,715,43,1
1,after,FETCH > IXSCAN,136,136,43,1


Index เรียง `province_code`, `vital_status` ซึ่งเป็น Equality ไว้ก่อน `birthdate` ที่เป็น Range ตามหลัก ESR; query ไม่มี Sort จึงไม่มีส่วน S
เป็น scalar compound index ไม่ใช่ multikey เพราะไม่ได้ใส่ array ใน index นี้
ก่อนสร้างยังมี index จังหวัด/อำเภอจาก load_data.py อยู่ จึงไม่จำเป็นต้องได้ COLLSCAN และรายงานแผนที่เกิดจริง
`status != removed`, `welf_eldery.code != WFEL9` และการคำนวณ `$expr` ยังต้องตรวจเอกสาร ส่วน index ไม่ช่วย stages หลัง lookup
เวลาเป็นการวัดครั้งเดียวบนข้อมูลเล็กและ cache อาจมีผล จึงใช้ docs/keys examined ประกอบ ไม่อ้างว่าเวลาเร็วขึ้นเสมอ

## Bonus B — Redesign the MSO-Logbook structure
### B1. ปัญหาที่พบจริง
1. **join ครัวเรือนทุกครั้ง:** Q2 และการจัดลำดับเคสต้อง lookup เพื่ออ่าน have_num/hh_type แม้เป็นข้อมูลที่อ่านร่วมกันประจำ เสี่ยงเสียคนที่ join ไม่พบถ้า unwind แบบปกติ
2. **ปัญหากระจายตาม domain:** O4 ต้องรวมเก้า array และ O5 ต้อง objectToArray/reduce; Q5 พบการแยกเก็บข้อเท็จจริง ปัญหา และ case management ทำให้ผู้ไม่พบบันทึกเบี้ยบางคนไม่มี PBAC8
3. **optional arrays/ข้อมูลคำนวณ:** Q3 ต้อง ifNull ซ้ำและตรวจยอดรายได้กับรายละเอียด; ค่า “ไม่มีโค้ด” ยังไม่ใช่สถานะยืนยันการรับเงินจริง

### B2. โครงสร้างใหม่: snapshot read model
สร้าง `mso_member_v2` จากข้อมูลเดิมโดยไม่แก้ต้นฉบับ เก็บ source IDs แบบ reference เพื่อสืบกลับและอัปเดตต้นทาง
embed `household_snapshot` เฉพาะ have_num/hh_type/status เพราะอ่านร่วมกับการคัดกรองเสมอ พร้อม snapshot_date เพื่อบอกความเก่า
รวมรหัสปัญหาเป็น `problem_codes` และ case management เป็น array เดียว `case_problems` ซึ่งเก็บ domain เดิมและ plans/activities
เก็บ `welfare_codes` และ `has_recorded_allowance` เป็นค่าคำนวณจากรหัสเดิม ไม่เติมข้อสรุปสิทธิหรือการรับเงินจริง
เก็บ `elderly_at_snapshot` และ age_years ภายใต้วันที่ snapshot ชัดเจน; ไม่ใช้ age นี้เป็นอายุปัจจุบัน

**ต้นทุน:** ข้อมูลซ้ำและเอกสารโตขึ้น ต้อง rebuild/incremental refresh เมื่อ household หรือ welfare เปลี่ยน
การแก้ source หลายเอกสารไม่ atomic ร่วมกับ read model จึงต้องมีงาน refresh และตรวจ reconciliation
ไม่ได้เก็บทุกฟิลด์เดิม; v2 เป็น materialized read model สำหรับการอ่านโจทย์นี้ ไม่ใช่ master database ใหม่

In [24]:
# Validator checks required types, two-digit province code and normalized problem codes.
validator = {'$jsonSchema': {
 'bsonType': 'object', 'required': ['source_member_id','snapshot_date','province_code','elderly_at_snapshot','has_recorded_allowance','problem_codes'],
 'properties': {
   'source_member_id': {'bsonType':'objectId'}, 'snapshot_date': {'bsonType':'date'},
   'province_code': {'bsonType':'string', 'pattern': '^[0-9]{2}$'},
   'elderly_at_snapshot': {'bsonType':'bool'}, 'has_recorded_allowance': {'bsonType':'bool'},
   'problem_codes': {'bsonType':'array', 'items': {'bsonType':'string'}}
 }}}
if 'mso_member_v2' not in db.list_collection_names():
    db.create_collection('mso_member_v2', validator=validator)
else:
    db.command('collMod', 'mso_member_v2', validator=validator, validationLevel='strict')
# Refresh only our derived collection; original three collections are unchanged.
db.mso_member_v2.delete_many({})
CASE_NORMALIZED = {'$reduce': {
 'input': {'$objectToArray': {'$ifNull': ['$prob_summary', {}]}}, 'initialValue': [],
 'in': {'$concatArrays': ['$$value', {'$map': {'input': '$$this.v', 'as': 'p', 'in': {'$mergeObjects': ['$$p', {'domain':'$$this.k'}]}}}]}
}}
WELFARE_FIELDS = ['welf_children_youth','welf_disability','welf_women','welf_eldery','welf_helpless_persons','welf_others']
migration = [
 {'$lookup': {'from':'mso_household','localField':'hh_id','foreignField':'_id','as':'hh'}},
 {'$unwind': {'path':'$hh','preserveNullAndEmptyArrays':True}},
 {'$project': {
   '_id':1, 'source_member_id':'$_id', 'source_household_id':'$hh_id', 'source_family_id':'$fam_id',
   'snapshot_date': {'$literal':SNAPSHOT}, 'province_code':1, 'amphur_code':1, 'birthdate':1, 'sex':1, 'status':1, 'vital_status':1,
   'age_years': AGE, 'elderly_at_snapshot': {'$gte':[AGE,60]},
   'has_recorded_allowance': has('welf_eldery.code','WFEL9'),
   'welfare_codes': {'$setUnion':[arr(p+'.code') for p in WELFARE_FIELDS]},
   'problem_codes': {'$setUnion':[arr(p+'.code') for p in PROB_FIELDS]},
   'case_problems': CASE_NORMALIZED,
   'household_snapshot': {'have_num': {'$ifNull':['$hh.have_num',None]}, 'hh_type': {'$ifNull':['$hh.hh_type.code',None]}, 'status': {'$ifNull':['$hh.status',None]}}
 }},
 {'$merge': {'into':'mso_member_v2','on':'_id','whenMatched':'replace','whenNotMatched':'insert'}}
]
list(db.mso_member.aggregate(migration))
db.mso_member_v2.create_index([('province_code',1),('vital_status',1),('elderly_at_snapshot',1),('has_recorded_allowance',1)], name='v2_target_scalar')
db.mso_member_v2.create_index('problem_codes', name='v2_problem_codes_multikey')
print('v2 documents:', db.mso_member_v2.count_documents({}))
sample = db.mso_member_v2.find_one({'province_code':'58','elderly_at_snapshot':True,'has_recorded_allowance':False})
# Actual migrated document; print concise sample with arrays retained.
print(json_util.dumps(sample, ensure_ascii=False, indent=2))
pass

v2 documents: 7518
{
  "_id": {
    "$oid": "63c52033477278f7cb002433"
  },
  "birthdate": {
    "$date": {
      "$numberLong": "-1099094400000"
    }
  },
  "sex": "ชาย",
  "vital_status": 0,
  "amphur_code": "5801",
  "province_code": "58",
  "status": "removed",
  "source_member_id": {
    "$oid": "63c52033477278f7cb002433"
  },
  "source_household_id": {
    "$oid": "63c52029add80c555b002431"
  },
  "source_family_id": {
    "$oid": "63c5202aa16b2ea92c002432"
  },
  "snapshot_date": {
    "$date": "2024-06-30T00:00:00Z"
  },
  "age_years": 89,
  "elderly_at_snapshot": true,
  "has_recorded_allowance": false,
  "welfare_codes": [],
  "problem_codes": [
    "PBAC8",
    "PBHE6",
    "PBOC7",
    "PBOT4"
  ],
  "case_problems": [
    {
      "code": "PBHE6",
      "desc": "เจ็บป่วยเรื้อรัง",
      "status": 0,
      "plans": [],
      "domain": "prob_health"
    },
    {
      "code": "PBOC7",
      "desc": "ไม่มีการออม",
      "status": 1,
      "plans": [
        {
          "sol_i

Index แรกเป็น scalar compound index สำหรับค้น target ส่วน `problem_codes` เป็น multikey แยกต่างหาก จึงไม่เสี่ยง compound index ที่มีหลาย array; validator บังคับรหัสจังหวัดและชนิด flag แต่ความสอดคล้องกับ source ต้องตรวจจากการ rebuild และ reconciliation เพิ่ม

### B3. รัน Q2 เดิมเทียบ v2

**คำถาม:** เมื่อฝัง household snapshot แล้ว ผลเปรียบเทียบได้เท่าเดิมโดยใช้ stages น้อยลงหรือไม่?

In [25]:
v2_flags = {'missing_allowance': {'$not':['$has_recorded_allowance']}}
q2_v2 = [
 {'$match': {**ACTIVE, 'elderly_at_snapshot':True, 'household_snapshot.status':{'$ne':'removed'}}},
 {'$set':v2_flags},
 {'$facet': {
   'house_number': rate_group({'$ifNull':['$household_snapshot.have_num','UNKNOWN']}),
   'household_type': rate_group({'$ifNull':['$household_snapshot.hh_type','UNKNOWN']})
 }}
]
new_result = list(db.mso_member_v2.aggregate(q2_v2))[0]
def canonical(records): return sorted(records, key=lambda r: str(r['_id']))
for key in housing:
    assert canonical(housing[key]) == canonical(new_result[key]), key
    print(key, '| exact result match'); display(pd.DataFrame(new_result[key]))
def stage_count(pipeline):
    return len(pipeline) + sum(stage_count(branch) for stage in pipeline if '$facet' in stage for branch in stage['$facet'].values())
comparison = pd.DataFrame([
 {'version':'original Q2','top_level_stages':len(q2),'all_stages_including_facet':stage_count(q2),'lookup_stages':1,'exact_match':True},
 {'version':'v2 Q2','top_level_stages':len(q2_v2),'all_stages_including_facet':stage_count(q2_v2),'lookup_stages':0,'exact_match':True}
])
display(comparison)
assert db.mso_member_v2.count_documents({}) == counts['mso_member']
pass

house_number | exact result match


,_id,elderly,gap,gap_pct
0,0,81,37,45.68
1,1,1378,205,14.88


household_type | exact result match


,_id,elderly,gap,gap_pct
0,HHT1,245,63,25.71
1,HHT3,1010,151,14.95
2,HHT2,204,28,13.73


,version,top_level_stages,all_stages_including_facet,lookup_stages,exact_match
0,original Q2,7,13,1,True
1,v2 Q2,3,9,0,True


**ตีความ:** ผล Q2 ตรงกันทุกกลุ่มและอัตรา โดยลด lookup ที่ต้องทำซ้ำในเวลาอ่าน; ประโยชน์นี้แลกกับเวลา migration พื้นที่เก็บ และความเสี่ยงข้อมูล snapshot ล้าสมัย จึงไม่สรุปว่าเร็วกว่าในทุก workload

## ข้อจำกัดและการตรวจงาน
ข้อมูลสังเคราะห์ไม่เป็นตัวแทนประเทศไทย; ผลเป็นความสัมพันธ์ ไม่ใช่เหตุและผล
โค้ดสวัสดิการและปัญหาที่หายไปไม่ยืนยันการไม่มีสิทธิ/การไม่ได้รับเงินจริง รวมทั้ง PBAC8 มีความหมายร่วมสำหรับเบี้ยพิการ
ข้อมูลรายได้ขาดได้และมี SUM_INCOME ไม่ตรงรายละเอียด debt เป็นรายจ่ายชำระต่อเดือน
อายุใช้สูตรปีตามโจทย์ และมี sensitivity เทียบอายุครบวันเกิด; ข้อมูลภายนอกเป็นรายปีที่ไม่ตรงวันที่ snapshot ทุกวัน
ข้อความอิสระสั้นหรือว่างทำให้ยังใช้ full-text/embedding ตรวจเหตุปัญหาได้จำกัด ต้องเก็บบันทึกเยี่ยมบ้านและผลยืนยันสิทธิที่มีคุณภาพก่อน

### Reproducibility and integrity checks

In [26]:
assert sum(x['n'] for x in dashboard['province']) == int(baseline.gap)
assert index_results.nReturned.nunique() == 1
assert all(canonical(housing[k]) == canonical(new_result[k]) for k in housing)
assert counts == {c:db[c].count_documents({}) for c in counts}
summary = {
 'name':'Anupat Suttilert','student_id':'6624650526','snapshot':'2024-06-30','runtime':runtime_info,
 'elderly':int(baseline.elderly),'gap':int(baseline.gap),'gap_pct':float(baseline.gap_pct),
 'disabled':int(alternative.disabled),'disabled_no_card':int(alternative.no_card),
 'pilot_people':pilot_n,'pilot_households':pilot_hh,'budget_baht':pilot_n*300,
 'housing':housing,'index_results':index_results.to_dict('records'),
 'external':external_context.to_dict('records'),
 'recording_gap':gap_recording.to_dict('records'), 'statement':statement,
 'age_sensitivity':age_sensitivity, 'v2_comparison':comparison.to_dict('records')
}
(ROOT/'output/verified_results.json').write_text(json_util.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
for key, frame in RESULTS.items(): frame.to_csv(ROOT/'output'/f'{key}.csv',index=False,encoding='utf-8-sig')
field_log.to_csv(ROOT/'output/field_log.csv',index=False,encoding='utf-8-sig')
print('PASS: counts, external joins, dashboard totals, targeting before/after and v2 exact equivalence.')
print('Task 1: 5 overview pipelines | Task 3: >5 evidence pipelines | fields:',len(field_log))
print('All original collection counts unchanged. No future monitoring results fabricated.')
pass

PASS: counts, external joins, dashboard totals, targeting before/after and v2 exact equivalence.
Task 1: 5 overview pipelines | Task 3: >5 evidence pipelines | fields: 30
All original collection counts unchanged. No future monitoring results fabricated.
